In [ ]:
# ---------------------------------------------------------------
# CAPÍTULO 13 — Ejercicio 1
# ---------------------------------------------------------------

import seaborn as sns
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
)
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score

# Dataset Titanic preparado
titanic = sns.load_dataset("titanic").dropna(subset=["embarked"])
titanic["age"] = titanic["age"].fillna(titanic["age"].median())
titanic["sex_cod"] = titanic["sex"].map({"male": 0, "female": 1})
X = titanic[["pclass", "sex_cod", "age", "fare"]].to_numpy()
y = titanic["survived"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

def make_pipe(clf):
    return Pipeline([("sc", StandardScaler()), ("clf", clf)])

combos = {
    "A: LR+KNN+DT": [
        ("lr", make_pipe(LogisticRegression(random_state=42))),
        ("knn", make_pipe(KNeighborsClassifier())),
        ("dt", make_pipe(DecisionTreeClassifier(random_state=42))),
    ],
    "B: RF+GB+SVC": [
        ("rf", RandomForestClassifier(random_state=42)),
        ("gb", GradientBoostingClassifier(random_state=42)),
        ("svc", make_pipe(SVC(probability=True, random_state=42))),
    ],
    "C: LR+RF+KNN+SVC+GB": [
        ("lr", make_pipe(LogisticRegression(random_state=42))),
        ("rf", make_pipe(RandomForestClassifier(n_estimators=100, random_state=42))),
        ("knn", make_pipe(KNeighborsClassifier())),
        ("svc", make_pipe(SVC(probability=True, random_state=42))),
        ("gb", make_pipe(GradientBoostingClassifier(random_state=42))),
    ],
}

for nombre, estimadores in combos.items():
    for voting in ["hard", "soft"]:
        vc = VotingClassifier(estimators=estimadores, voting=voting)
        scores = cross_val_score(vc, X_train, y_train, cv=5)
        print(f"{nombre} | {voting:4s} | CV={scores.mean():.4f} \u00b1 {scores.std():.4f}")

In [ ]:
# ---------------------------------------------------------------
# CAPÍTULO 13 — Ejercicio 2
# ---------------------------------------------------------------

from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score

modelos_base = [
    ("lr",  Pipeline([("sc", StandardScaler()),
            ("clf", LogisticRegression(random_state=42))])),
    ("rf",  RandomForestClassifier(n_estimators=100, random_state=42)),
    ("gb",  GradientBoostingClassifier(random_state=42)),
    ("knn", Pipeline([("sc", StandardScaler()),
            ("clf", KNeighborsClassifier())])),
]

meta_modelos = {
    "LR":  LogisticRegression(),
    "RF":  RandomForestClassifier(n_estimators=50, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=3),
}

for nombre, meta in meta_modelos.items():
    stk = StackingClassifier(estimators=modelos_base, final_estimator=meta, cv=5, n_jobs=-1)

    # CV accuracy
    scores = cross_val_score(stk, X_train, y_train, cv=5)
    print(f"Meta={nombre:4s} | CV={scores.mean():.4f} ± {scores.std():.4f}")

    # Test accuracy
    stk.fit(X_train, y_train)
    y_pred = stk.predict(X_test)
    print(f"          | Test={accuracy_score(y_test, y_pred):.4f}\n")


In [ ]:
# ---------------------------------------------------------------
# CAPÍTULO 13 — Ejercicio 3
# ---------------------------------------------------------------

import seaborn as sns
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import (StackingClassifier,
    RandomForestClassifier, GradientBoostingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, train_test_split

df_tit = sns.load_dataset("titanic")
cols_num = ["age", "fare", "sibsp", "parch"]
cols_cat = ["sex", "embarked", "class"]
X_tit = df_tit[cols_num + cols_cat]
y_tit = df_tit["survived"]

preprocesador = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc",  StandardScaler()),
    ]), cols_num),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), cols_cat),
])

def pipe_base(clf):
    return Pipeline([("prep", preprocesador), ("clf", clf)])

modelos_capa1 = [
    ("lr",  pipe_base(LogisticRegression(random_state=42))),
    ("rf",  pipe_base(RandomForestClassifier(
            n_estimators=100, random_state=42))),
    ("gb",  pipe_base(GradientBoostingClassifier(random_state=42))),
    ("knn", pipe_base(KNeighborsClassifier(n_neighbors=5))),
]

stacking = StackingClassifier(
    estimators=modelos_capa1,
    final_estimator=LogisticRegression(),
    cv=5, n_jobs=-1
)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_tit, y_tit, test_size=0.2, random_state=42, stratify=y_tit
)
scores = cross_val_score(stacking, X_tr, y_tr, cv=5)
print(f"Stacking CV: {scores.mean():.4f} ± {scores.std():.4f}")

# 1. Comparar con modelos individuales
print("Evaluación modelos individuales vs Ensemble:")
for nombre, clf in modelos_capa1:
    scores = cross_val_score(clf, X_tr, y_tr, cv=5)
    print(f"{nombre:4s} | CV={scores.mean():.4f}")

# 2. Entrenar ensemble completo e inspeccionar meta-modelo
stacking.fit(X_tr, y_tr)
print("\nCoeficientes del meta-modelo (LogisticRegression):")
# El meta-modelo aprende 1 coeficiente por modelo base
for (nombre, _), coef in zip(modelos_capa1, stacking.final_estimator_.coef_[0]):
    print(f"{nombre:4s}: {coef:+.4f}")
